# TradeSense · Component 03 · Fusion v0 (notebook 35): one fused score from the three bar-level models (CPU, no GPU)

**Question (protocol F0, stored in the project doc `protocol_fusion_v0.md`, written before any fusion result):** the three final models at 2 episodes/day each give a union recall around 0.8 but about 4.2 genuine episodes/day. Can one fused score keep a similar recall at **2** episodes/day, and does it beat the best single model at the same burden?

What this notebook does
1. checks that notebooks 28–31 finished (it reads `declared_final_set_v1.json`); TEST-B / TEST-3 are **never loaded**;
2. unit tests (causal scaling, fit rows, out-of-fold purge, recall by a plain loop, agreement with notebook 31);
3. builds the 8 fusion features from the saved scores, fits the stacker candidates (logistic regression, shallow trees) with **out-of-fold** scores on DEV, selects by the pre-registered rule;
4. DEV comparison: fusion vs each single model vs the union (same bars, same rule);
5. CAL-3 evaluation (the stacker fitted on VAL-3 only, applied unchanged): recall at budgets 1/2/4, realised genuine episodes/day, per family, bootstrap intervals, H-F1…H-F4;
6. per-model contribution of the logistic stacker per family (the explainable part), saves fused scores and `declared_fusion_v0.json`.

Protocol text (frozen by hash in cell 1):
# TradeSense C03 – Protocol F0 (fusion v0 of the three bar-level models) – 2026-10-14
Written BEFORE any fusion result exists. Frozen by sha256 when notebook 35 runs. Supersedes nothing: Protocol B2 and amendments A1–A3 stay as they are; the TGNN joins fusion in a later protocol (F1), because its scores exist only for the eight order-book days.

## 1. Question
Taking the three final models of `declared_final_set_v1.json` at 2 alarm episodes/day each gives a union recall of about 0.82 (VAL-3, old families) but 4.2 genuine episodes/day. Can ONE fused score reach a similar recall at 2 episodes/day (and 4)? And is it better than the best single model at the same burden?

## 2. Inputs and roles (no label of CAL or TEST is used to fit anything)
- Base scores: the saved scores of the final models (normalisation as declared). XGBoost and Transformer scores are margins (logits); AE-ctx fmax is non-negative and heavy-tailed, so it enters as log(1 + score).
- Causal input scaling: z = (s − rolling 7-day median) / (rolling 7-day IQR/1.349) of the model's own score at bars strictly before t (minimum 1,440 bars, then the frozen DEV genuine median/scale; scale floor 0.1 × DEV scale). For an injected series the rolling statistics come from the GENUINE series at the same bars (the T1c idea of amendment A1: dense synthetic events must not contaminate the history). On genuine series the statistics come from the series itself.
- Fusion features per bar (8): z of the three models, max and mean of the three z, and the maximum of each z over the last 5 bars.
- Fit rows: VAL-3 (both replicas): positives = bars of the six old families inside events; negatives = clean bars of replica 0 (no non-clean bar in the previous 16 bars). The two new families (momentum_ignition, slow_accumulation) are never used for fitting or selection.
- Out-of-fold scores: DEV (bars 90720..129599) is cut into 4 contiguous blocks; the stacker for a block is fitted on the other blocks with a 240-bar purge on each side of the block. All DEV selection numbers use these out-of-fold scores only. The final stacker is fitted on all VAL-3 rows and is applied to CAL-3 without refitting.
- Honest limitation: the base models were selected on VAL-3, so VAL-3 is not entirely untouched by them; CAL-3 is the clean evaluation.

## 3. Stacker candidates and selection (DEV out-of-fold only)
- Logistic regression (L2), C ∈ {0.01, 0.1, 1}; shallow boosted trees (XGBoost, depth 2, 200 trees, learning rate 0.05).
- Metric: out-of-fold recall (macro over the six old families, mean of BTC/ETH) at 2 episodes/day with the T1c rule; tolerance 0.05; simplest first (logistic before trees, smaller C first). Selection happens once; the choice is then frozen.

## 4. Threshold and burden
- T1 rule exactly as in B2: rolling 7-day own-score quantile, q per budget frozen from the genuine DEV bars (out-of-fold fused score); budgets 1, 2, 4 episodes/day, primary 2. On injected series T1c (rolling threshold of the genuine series). Realised genuine episodes/day on DEV and CAL are reported next to every recall.
- For the DEV comparison the single models and their union are re-evaluated on the same DEV bars (the first 1,440 DEV bars are dropped for everyone, because the rolling threshold needs history inside the evaluated stretch).

## 5. Hypotheses (pre-registered, reported as failed if they fail)
- H-F1 (burden): on CAL-3, fusion at budget 2 has recall (six old families) not more than 0.05 below the union of the three models (each at 2/day) while its realised genuine CAL episodes/day is at most 60% of the union's.
- H-F2 (beats the best single model): on CAL-3, fusion at budget 2 beats the best single final model at budget 2 by at least 0.03 recall (six old families), with the paired event-bootstrap 95% interval of the difference above 0.
- H-F3 (unseen families): on CAL-3, the two new families, fusion recall at budget 2 is not more than 0.05 below the best single model.
- H-F4 (descriptive, no threshold): recall of fusion versus the best single model per family (including abnormal_volume_burst); no decision attached.

## 6. Explainability and uncertainty hooks (reported, not decided here)
- Logistic stacker: the contribution of each input to a score is coefficient × feature (additive, exact), reported as per-model contribution for alerts. Per-feature SHAP of the base models and the conformal interval of the Integrity Score are the next protocol (F1).

## 7. Test discipline
Stacker selected on DEV out-of-fold → CAL-3 and genuine CAL evaluated once → `declared_fusion_v0.json` written → TEST-3 evaluated in notebook 32 with the same frozen stacker. No change after CAL.

## 8. Limitations stated in advance
Labels are synthetic mechanisms; the stacker learns from synthetic positives of six families; two symbols; the base models' VAL-3 scores are out of sample but VAL-3 was used for their selection; genuine bars are unlabelled.

In [ ]:
import os, json, glob, time, hashlib, warnings, sys, datetime, subprocess
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
try:
    import xgboost as xgb
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'xgboost']); import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import joblib
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets'))
L3 = BASE/'layer3_v3'; OUT = BASE/'models_b2_v1'; FUS = OUT/'fusion'; FUS.mkdir(parents=True, exist_ok=True)
SYMBOLS = ['BTCUSDT', 'ETHUSDT']; MODES = ['N0', 'N1']
MOCK = os.environ.get('TS_MOCK') == '1'; NBOOT = 200 if MOCK else 1000; NTHREAD = max(1, os.cpu_count() or 1)
LINES = []
def say(s=''): print(s); LINES.append(str(s))
def sha_file(p, n=16): return hashlib.sha256(Path(p).read_bytes()).hexdigest()[:n]
hf = L3/'protocol_bar_models_v2.sha256'; pf = L3/'protocol_bar_models_v2.md'
assert hf.exists() and pf.exists(), 'run notebook 27 first'
ph = hashlib.sha256(pf.read_text().encode()).hexdigest(); assert ph == hf.read_text().split()[0], 'frozen protocol B2 was changed'
for f in ['declared_final_set_v1.json', 'cal3_recall.csv', 'b2_step5_share_block.txt']:
    assert (OUT/f).exists(), f'{f} missing: run notebook 31 to the end first'
PROTO_TEXT = r"""# TradeSense C03 – Protocol F0 (fusion v0 of the three bar-level models) – 2026-10-14
Written BEFORE any fusion result exists. Frozen by sha256 when notebook 35 runs. Supersedes nothing: Protocol B2 and amendments A1–A3 stay as they are; the TGNN joins fusion in a later protocol (F1), because its scores exist only for the eight order-book days.

## 1. Question
Taking the three final models of `declared_final_set_v1.json` at 2 alarm episodes/day each gives a union recall of about 0.82 (VAL-3, old families) but 4.2 genuine episodes/day. Can ONE fused score reach a similar recall at 2 episodes/day (and 4)? And is it better than the best single model at the same burden?

## 2. Inputs and roles (no label of CAL or TEST is used to fit anything)
- Base scores: the saved scores of the final models (normalisation as declared). XGBoost and Transformer scores are margins (logits); AE-ctx fmax is non-negative and heavy-tailed, so it enters as log(1 + score).
- Causal input scaling: z = (s − rolling 7-day median) / (rolling 7-day IQR/1.349) of the model's own score at bars strictly before t (minimum 1,440 bars, then the frozen DEV genuine median/scale; scale floor 0.1 × DEV scale). For an injected series the rolling statistics come from the GENUINE series at the same bars (the T1c idea of amendment A1: dense synthetic events must not contaminate the history). On genuine series the statistics come from the series itself.
- Fusion features per bar (8): z of the three models, max and mean of the three z, and the maximum of each z over the last 5 bars.
- Fit rows: VAL-3 (both replicas): positives = bars of the six old families inside events; negatives = clean bars of replica 0 (no non-clean bar in the previous 16 bars). The two new families (momentum_ignition, slow_accumulation) are never used for fitting or selection.
- Out-of-fold scores: DEV (bars 90720..129599) is cut into 4 contiguous blocks; the stacker for a block is fitted on the other blocks with a 240-bar purge on each side of the block. All DEV selection numbers use these out-of-fold scores only. The final stacker is fitted on all VAL-3 rows and is applied to CAL-3 without refitting.
- Honest limitation: the base models were selected on VAL-3, so VAL-3 is not entirely untouched by them; CAL-3 is the clean evaluation.

## 3. Stacker candidates and selection (DEV out-of-fold only)
- Logistic regression (L2), C ∈ {0.01, 0.1, 1}; shallow boosted trees (XGBoost, depth 2, 200 trees, learning rate 0.05).
- Metric: out-of-fold recall (macro over the six old families, mean of BTC/ETH) at 2 episodes/day with the T1c rule; tolerance 0.05; simplest first (logistic before trees, smaller C first). Selection happens once; the choice is then frozen.

## 4. Threshold and burden
- T1 rule exactly as in B2: rolling 7-day own-score quantile, q per budget frozen from the genuine DEV bars (out-of-fold fused score); budgets 1, 2, 4 episodes/day, primary 2. On injected series T1c (rolling threshold of the genuine series). Realised genuine episodes/day on DEV and CAL are reported next to every recall.
- For the DEV comparison the single models and their union are re-evaluated on the same DEV bars (the first 1,440 DEV bars are dropped for everyone, because the rolling threshold needs history inside the evaluated stretch).

## 5. Hypotheses (pre-registered, reported as failed if they fail)
- H-F1 (burden): on CAL-3, fusion at budget 2 has recall (six old families) not more than 0.05 below the union of the three models (each at 2/day) while its realised genuine CAL episodes/day is at most 60% of the union's.
- H-F2 (beats the best single model): on CAL-3, fusion at budget 2 beats the best single final model at budget 2 by at least 0.03 recall (six old families), with the paired event-bootstrap 95% interval of the difference above 0.
- H-F3 (unseen families): on CAL-3, the two new families, fusion recall at budget 2 is not more than 0.05 below the best single model.
- H-F4 (descriptive, no threshold): recall of fusion versus the best single model per family (including abnormal_volume_burst); no decision attached.

## 6. Explainability and uncertainty hooks (reported, not decided here)
- Logistic stacker: the contribution of each input to a score is coefficient × feature (additive, exact), reported as per-model contribution for alerts. Per-feature SHAP of the base models and the conformal interval of the Integrity Score are the next protocol (F1).

## 7. Test discipline
Stacker selected on DEV out-of-fold → CAL-3 and genuine CAL evaluated once → `declared_fusion_v0.json` written → TEST-3 evaluated in notebook 32 with the same frozen stacker. No change after CAL.

## 8. Limitations stated in advance
Labels are synthetic mechanisms; the stacker learns from synthetic positives of six families; two symbols; the base models' VAL-3 scores are out of sample but VAL-3 was used for their selection; genuine bars are unlabelled.
"""
pp = FUS/'protocol_fusion_v0.md'; pp.write_text(PROTO_TEXT); PH = hashlib.sha256(PROTO_TEXT.encode()).hexdigest()
say('protocol B2 intact, sha256 ' + ph[:16] + '; fusion protocol F0 sha256 ' + PH[:16] + ('  [MOCK RUN]' if MOCK else ''))

Mounted at /content/drive
protocol B2 intact, sha256 443f99d5faf5929f; fusion protocol F0 sha256 b1d84483c102bf65


## 1 · Library (same file as notebooks 28-31)

In [ ]:
_old_lib = (OUT/'b2_lib.py').read_text() if (OUT/'b2_lib.py').exists() else None
open("b2_lib.py", "w").write(r'''
"""TradeSense C03 - Protocol B2 shared library: normalisation N0/N1, conditional autoencoder (AE-ctx), burden thresholds T0/T1, recall."""
import numpy as np, pandas as pd, torch, torch.nn as nn, time, json, hashlib
from numpy.lib.stride_tricks import sliding_window_view

FEATS = ['log_returns', 'trade_flow_imbalance', 'hl_spread_ratio', 'rush_order_density', 'volume_acceleration', 'large_trade_count']
LOGC = [3, 5]; WIN7 = 10080; MINP = 1440; CTX = 15; WARM = 30; GAPM = 5; CLIP = 20.0
BUDGETS = [1, 2, 4]; PRIMARY = 2; QGRID = [0.99, 0.993, 0.995, 0.997, 0.998, 0.999, 0.9993, 0.9995, 0.9997, 0.9998, 0.9999, 0.99995]
FAMS = {1: 'pump_and_dump_like', 2: 'fake_breakout_like', 3: 'abnormal_volume_burst', 4: 'trade_flow_imbalance', 5: 'wash_like_aggregate_proxy',
        6: 'liquidity_stress_proxy', 7: 'momentum_ignition_like', 8: 'slow_accumulation_like'}


# ---------------------------------------------------------------- normalisation
def to_x(F):
    x = np.asarray(F[FEATS] if hasattr(F, 'columns') else F, dtype='float64').copy()
    for j in LOGC: x[:, j] = np.log1p(x[:, j])
    return x


def fit_n0(F_train):
    """median / (IQR/1.349) on log1p-transformed Layer 1 TRAIN bars (caller passes bars >= 15 only)."""
    x = to_x(F_train); med = np.median(x, axis=0); q75, q25 = np.percentile(x, [75, 25], axis=0); return {'median': med, 'scale': (q75 - q25) / 1.349}


def normalise(F, mode, n0):
    x = to_x(F)
    if mode == 'N0': med, scl = n0['median'][None, :], n0['scale'][None, :]
    else:
        df = pd.DataFrame(x); r = df.rolling(WIN7, min_periods=MINP)
        med = r.median().shift(1).to_numpy(); q75 = r.quantile(0.75).shift(1).to_numpy(); q25 = r.quantile(0.25).shift(1).to_numpy()
        scl = np.maximum((q75 - q25) / 1.349, 0.1 * n0['scale'][None, :])
        bad = np.isnan(med) | np.isnan(scl); med = np.where(bad, n0['median'][None, :], med); scl = np.where(bad, n0['scale'][None, :], scl)
    return np.clip((x - med) / scl, -CLIP, CLIP)


def context(z):
    """causal context: mean and std (ddof 1) of the previous CTX bars; rows < CTX are zero-filled and never used."""
    n = len(z); m = np.zeros_like(z); s = np.zeros_like(z)
    w = sliding_window_view(z, CTX, axis=0)[: n - CTX]            # window k = bars k..k+14 -> bar k+15
    m[CTX:] = w.mean(axis=2); s[CTX:] = w.std(axis=2, ddof=1); return m, s


# ---------------------------------------------------------------- conditional autoencoder
class AECtx(nn.Module):
    def __init__(self, d=6, b=3):
        super().__init__(); self.enc = nn.Sequential(nn.Linear(d + 2 * d, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, b))
        self.dec = nn.Sequential(nn.Linear(b + 2 * d, 32), nn.ReLU(), nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, d))

    def forward(self, z, cm, cs):
        c = torch.cat([cm, cs], 1); return self.dec(torch.cat([self.enc(torch.cat([z, c], 1)), c], 1))


def train_ae(z, idx, seed, max_epochs=150, patience=10, batch=512, log=None):
    """z normalised genuine series; idx = bar indices allowed for fitting (valid, inside Layer 1 TRAIN). 90% fit / 10% early stop, 15-bar gap."""
    torch.manual_seed(seed); rng = np.random.default_rng(seed); cm, cs = context(z)
    idx = np.sort(idx); cut = int(0.9 * len(idx)); fit = idx[:cut]; val = idx[idx > idx[cut] + CTX]
    T = lambda a: torch.tensor(a, dtype=torch.float32)
    Zf, Mf, Sf = T(z[fit]), T(cm[fit]), T(cs[fit]); Zv, Mv, Sv = T(z[val]), T(cm[val]), T(cs[val])
    net = AECtx(); opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-5); best, bad, state = 1e18, 0, None
    for ep in range(max_epochs):
        net.train(); perm = torch.tensor(rng.permutation(len(fit)))
        for i in range(0, len(perm), batch):
            b = perm[i:i + batch]; opt.zero_grad(); loss = ((net(Zf[b], Mf[b], Sf[b]) - Zf[b]) ** 2).mean(); loss.backward(); opt.step()
        net.eval()
        with torch.no_grad(): v = float(((net(Zv, Mv, Sv) - Zv) ** 2).mean())
        if v < best - 1e-6: best, bad, state = v, 0, {k: x.clone() for k, x in net.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: break
    net.load_state_dict(state); net.eval(); return net, {'epochs': ep + 1, 'best_val': best, 'fit_rows': int(len(fit)), 'val_rows': int(len(val)), 'fit_min': int(fit.min()), 'fit_max': int(fit.max())}


def ae_errors(net, z):
    cm, cs = context(z); T = lambda a: torch.tensor(a, dtype=torch.float32)
    with torch.no_grad(): rec = net(T(z), T(cm), T(cs)).numpy()
    return (z - rec) ** 2


def p99_scale(err, idx): return np.percentile(err[idx], 99, axis=0) + 1e-12


def ae_columns(err, scale):
    """8 columns: fmax score, fmean score, 6 per-feature errors / TRAIN p99 scale."""
    r = err / scale[None, :]; return np.column_stack([r.max(axis=1), err.mean(axis=1), r])


# ---------------------------------------------------------------- burden: episodes, thresholds, recall
def episodes(alarm, gap=GAPM):
    a = np.flatnonzero(alarm)
    if len(a) == 0: return 0
    return int(1 + np.sum(np.diff(a) > gap))


def eps_per_day(alarm, valid):
    a = alarm & valid; days = valid.sum() / 1440.0
    return episodes(a) / max(days, 1e-9)


def valid_mask(n, lo, hi): m = np.zeros(n, bool); m[max(lo, WARM):hi] = True; return m


def t0_threshold(score, valid, budget):
    """lowest quantile-grid threshold (descending probability grid, lowering the threshold step by step) with episodes/day <= budget."""
    s = score[valid]; best = None
    for q in sorted(QGRID, reverse=True):
        thr = np.quantile(s, q)
        if eps_per_day(score > thr, valid) <= budget: best = thr
        else: break
    return best if best is not None else np.quantile(s, max(QGRID))


def rolling_thr(score, q):
    return pd.Series(score).rolling(WIN7, min_periods=MINP).quantile(q).shift(1).to_numpy()


def t1_q(score, valid, budget):
    """smallest quantile level q (lowest rolling threshold) with episodes/day <= budget on the genuine DEV bars."""
    for q in QGRID:
        thr = rolling_thr(score, q); alarm = (score > np.where(np.isnan(thr), np.inf, thr))
        if eps_per_day(alarm, valid) <= budget: return q
    return QGRID[-1]


def alarms(score, rule, par):
    if rule == 'T0': return score > par
    thr = rolling_thr(score, par); return score > np.where(np.isnan(thr), np.inf, thr)


def recall_by_family(alarm, ev, seg_ok=None):
    """ev rows: t0, dur, family, (replica); detected if an alarm falls in [t0, t0+dur-1]."""
    c = np.concatenate([[0], np.cumsum(alarm.astype(np.int64))]); out = {}
    for fam, sub in ev.groupby('family'):
        hit = [(c[r.t0 + r.dur] - c[r.t0]) > 0 for r in sub.itertuples()]; out[int(fam)] = (int(np.sum(hit)), int(len(hit)))
    return out


def bootstrap_ci(hits, n=1000, seed=0):
    rng = np.random.default_rng(seed); h = np.asarray(hits, float)
    if len(h) == 0: return (np.nan, np.nan)
    b = [h[rng.integers(0, len(h), len(h))].mean() for _ in range(n)]; return (float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5)))


def macro(rec, fams):
    v = [rec[f][0] / rec[f][1] for f in fams if f in rec and rec[f][1] > 0]; return float(np.mean(v)) if v else float('nan')


def sha_bytes(b): return hashlib.sha256(b).hexdigest()

''')

import sys, importlib; sys.path.insert(0, '.'); import b2_lib as B; importlib.reload(B); (OUT/'b2_lib.py').write_text(Path('b2_lib.py').read_text())
say('library sha256 ' + sha_file('b2_lib.py'))
assert _old_lib is None or _old_lib.strip() == Path('b2_lib.py').read_text().strip(), 'b2_lib.py differs from the copy written by notebook 28'

library sha256 1b0b46630763c944


## 2 · Data (saved scores, events, labels)

In [ ]:
DF = json.load(open(OUT/'declared_final_set_v1.json')); MODELS = list(DF['models']); FNORM = {m: DF['models'][m]['normalisation'] for m in MODELS}
FILES = {'AE-ctx fmax': 'ae_ctx', 'XGBoost v2': 'xgb_v2', 'Transformer v2': 'transformer_v2'}; assert set(MODELS) == set(FILES)
AOLEN = 31 * 1440; CAL_LO = 1440; PRIM = B.PRIMARY; BUD = B.BUDGETS; DEV_LO, DEV_HI = 90720, 129600
L1V = B.valid_mask(129600, DEV_LO, DEV_HI); CALV = B.valid_mask(AOLEN, CAL_LO, AOLEN)
VD = np.zeros(129600, bool); VD[DEV_LO + 1440:DEV_HI] = True; VD[:B.WARM] = False             # DEV bars used for the fair DEV comparison (first 1,440 DEV bars dropped for everyone)
OLD, NEW = [1, 2, 3, 4, 5, 6], [7, 8]; ALLF = OLD + NEW
def nanw(a): a = np.asarray(a, 'float64').copy(); a[:B.WARM] = np.nan; return a
SCR = {}; EV = {}; LAB = {}
for s in SYMBOLS:
    for name in ['VAL', 'CAL']: EV[(s, name)] = pd.read_csv(L3/f'l3v3_{s}_{name}_events.csv')
    f = pd.read_parquet(L3/f'l3v3_{s}_VAL.parquet', columns=['label', 'in_seg', 'replica'])
    for r, g in f.groupby('replica'): LAB[(s, int(r))] = (g.label.to_numpy().astype(int), g.in_seg.to_numpy().astype(bool))
    for model in MODELS:
        z = np.load(OUT/'scores'/f'{FILES[model]}_{s}_{FNORM[model]}.npz'); assert not any('TEST' in k for k in z.files); d = {}
        for k in z.files:
            if k == 'scale' or k.startswith('TRAIN'): continue
            a = z[k]; a = a[:, 0] if a.ndim == 2 else a
            d[k] = nanw(a[:AOLEN] if k.startswith('CAL') else a)
        assert len(d['GEN_L1']) == 129600 and len(d['GEN_AO']) == AOLEN and len(d['CAL_r0']) == AOLEN; SCR[(model, s)] = d
def event_sets(name):
    out = []
    for s in SYMBOLS:
        ev = EV[(s, name)]
        for r in sorted(ev.replica.unique()): out.append((s, f'{name}_r{int(r)}', ev[ev.replica == r].reset_index(drop=True)))
    return out
ESETS = {'VAL': event_sets('VAL'), 'CAL': event_sets('CAL')}
META = {n: pd.concat([pd.DataFrame({'sym': s, 'fam': ev.family.to_numpy(), 't0': ev.t0.to_numpy()}) for s, k, ev in ESETS[n]], ignore_index=True) for n in ESETS}
for s, k, ev in ESETS['CAL']: assert ev.t0.min() >= CAL_LO and (ev.t0 + ev.dur).max() <= AOLEN
def event_hits(a, ev):
    c = np.concatenate([[0], np.cumsum(np.nan_to_num(a).astype(np.int64))]); t0 = ev.t0.to_numpy(); du = ev.dur.to_numpy(); return (c[t0 + du] - c[t0]) > 0
def macro_of(h, name, fams, sel=None):
    m = META[name]; v = []
    for s in SYMBOLS:
        k = ((m.sym == s) & m.fam.isin(fams)).to_numpy().copy()
        if sel is not None: k &= sel
        d = pd.DataFrame({'fam': m.fam[k].to_numpy(), 'h': h[k]}); v.append(d.groupby('fam').h.mean().mean())
    return float(np.mean(v))
def fam_rates(h, name, sel=None):
    m = META[name]; k = np.ones(len(m), bool) if sel is None else sel; return pd.DataFrame({'sym': m.sym[k], 'fam': m.fam[k], 'h': h[k]}).groupby(['sym', 'fam']).h.mean().groupby('fam').mean()
def groups_of(name, fams, sel=None):
    m = META[name]; out = []
    for s in SYMBOLS:
        gs = []
        for f in fams:
            k = ((m.sym == s) & (m.fam == f)).to_numpy().copy()
            if sel is not None: k &= sel
            idx = np.flatnonzero(k)
            if len(idx): gs.append(idx)
        out.append(gs)
    return out
def boot_paired(h_list, name, fams, sel=None, n=None, seed=0):
    n = n or NBOOT; G = groups_of(name, fams, sel); rg = np.random.default_rng(seed); out = np.zeros((n, len(h_list)))
    for i in range(n):
        acc = []
        for gs in G:
            fam_means = []
            for idx in gs:
                pick = idx[rg.integers(0, len(idx), len(idx))]; fam_means.append([h[pick].mean() for h in h_list])
            acc.append(np.mean(fam_means, axis=0))
        out[i] = np.mean(acc, axis=0)
    return out
def ci(x): return f'{np.percentile(x, 2.5):.2f}-{np.percentile(x, 97.5):.2f}'
say('data ready: models ' + str(FNORM) + '; DEV evaluation bars ' + str(int(VD.sum())) + '; CAL events per symbol ' + str(META['CAL'].groupby('sym').size().to_dict()))

data ready: models {'AE-ctx fmax': 'N1', 'XGBoost v2': 'N0', 'Transformer v2': 'N1'}; DEV evaluation bars 37440; CAL events per symbol {'BTCUSDT': 200, 'ETHUSDT': 200}


## 3 · Fusion features and fit rows

In [ ]:
TRF = {'AE-ctx fmax': lambda x: np.log1p(np.maximum(x, 0)), 'XGBoost v2': lambda x: x, 'Transformer v2': lambda x: x}     # XGBoost / Transformer scores are margins (logits); AE fmax is heavy-tailed
STAT = {}
def stats(model, s, ref):
    k = (model, s, ref)
    if k not in STAT:
        x = TRF[model](SCR[(model, s)][ref]); r = pd.Series(x).rolling(10080, min_periods=1440)
        med = r.median().shift(1).to_numpy(); q75 = r.quantile(0.75).shift(1).to_numpy(); q25 = r.quantile(0.25).shift(1).to_numpy()
        g = TRF[model](SCR[(model, s)]['GEN_L1'])[L1V]; gmed = float(np.nanmedian(g)); gsc = float((np.nanpercentile(g, 75) - np.nanpercentile(g, 25)) / 1.349)
        scl = np.maximum((q75 - q25) / 1.349, 0.1 * gsc); bad = np.isnan(med) | np.isnan(scl); STAT[k] = (np.where(bad, gmed, med), np.where(bad, gsc, scl))
    return STAT[k]
def Zof(model, s, key, ref):
    med, scl = stats(model, s, ref); return np.clip((TRF[model](SCR[(model, s)][key]) - med) / scl, -15, 15)
def feats(s, key, ref):
    Zs = np.column_stack([Zof(m, s, key, ref) for m in MODELS]); roll = np.column_stack([pd.Series(Zs[:, i]).rolling(5, min_periods=1).max().to_numpy() for i in range(len(MODELS))])
    return np.column_stack([Zs, Zs.max(axis=1), Zs.mean(axis=1), roll]).astype(np.float32)
FNAMES = ['z ' + m for m in MODELS] + ['z max', 'z mean'] + ['max5 ' + m for m in MODELS]
FX = {}
for s in SYMBOLS:
    FX[(s, 'GEN_L1')] = feats(s, 'GEN_L1', 'GEN_L1')
    for r in sorted({int(k.split('_r')[1]) for k in SCR[(MODELS[0], s)] if k.startswith('VAL')}): FX[(s, f'VAL_r{r}')] = feats(s, f'VAL_r{r}', 'GEN_L1')
    FX[(s, 'GEN_AO')] = feats(s, 'GEN_AO', 'GEN_AO'); FX[(s, 'CAL_r0')] = feats(s, 'CAL_r0', 'GEN_AO')
say('fusion features ' + str(FNAMES) + '; example (BTC, genuine DEV) median ' + str(np.round(np.nanmedian(FX[('BTCUSDT', 'GEN_L1')][L1V], axis=0), 2)))
# ---------- fit rows (VAL-3 only; six old families as positives; clean negatives of replica 0; new families never used)
LOOK = 16; ROWS = []
for (s, r), (lab, inseg) in LAB.items():
    n = len(lab); idx = np.arange(n); c = np.concatenate([[0], np.cumsum((lab != 0).astype(np.int64))])
    clean = ((c[idx + 1] - c[np.maximum(idx - LOOK, 0)]) == 0) & inseg & (idx >= B.WARM); pos = (lab > 0) & (lab <= 6) & inseg & (idx >= B.WARM)
    m = pos | (clean if r == 0 else np.zeros(n, bool)); m &= ~np.isnan(FX[(s, f'VAL_r{r}')]).any(axis=1)
    ii = np.flatnonzero(m); ROWS.append(pd.DataFrame({'sym': s, 'rep': r, 'bar': ii, 'y': pos[ii].astype(int), 'lab': lab[ii]}))
ROWS = pd.concat(ROWS, ignore_index=True); assert ROWS.bar.min() >= DEV_LO and ROWS.bar.max() < DEV_HI and ROWS[ROWS.y == 1].lab.between(1, 6).all() and not ROWS.lab.isin([7, 8]).any()
def XY(rows):
    X = np.vstack([FX[(s, f'VAL_r{r}')][g.bar.to_numpy()] for (s, r), g in rows.groupby(['sym', 'rep'], sort=False)]); y = np.concatenate([g.y.to_numpy() for _, g in rows.groupby(['sym', 'rep'], sort=False)]); return X, y
say('fit rows: ' + str(int(ROWS.y.sum())) + ' positive bars (old families), ' + str(int((ROWS.y == 0).sum())) + ' clean negatives; new families excluded')

fusion features ['z AE-ctx fmax', 'z XGBoost v2', 'z Transformer v2', 'z max', 'z mean', 'max5 AE-ctx fmax', 'max5 XGBoost v2', 'max5 Transformer v2']; example (BTC, genuine DEV) median [-0.    0.04  0.01  0.74  0.09  1.53  0.79  0.79]
fit rows: 9286 positive bars (old families), 60015 clean negatives; new families excluded


## 4 · Unit tests

In [ ]:
# T01 causal scaling: changing scores at or after bar t does not change z before t
mm = MODELS[1]; s0 = SYMBOLS[0]; save = SCR[(mm, s0)]['GEN_AO'].copy(); z0 = Zof(mm, s0, 'GEN_AO', 'GEN_AO')[:20000].copy(); STAT.pop((mm, s0, 'GEN_AO'))
SCR[(mm, s0)]['GEN_AO'] = save.copy(); SCR[(mm, s0)]['GEN_AO'][20000:] = np.nanmax(save) * 5; z1 = Zof(mm, s0, 'GEN_AO', 'GEN_AO')[:20000]
SCR[(mm, s0)]['GEN_AO'] = save; STAT.pop((mm, s0, 'GEN_AO')); assert np.allclose(np.nan_to_num(z0), np.nan_to_num(z1)); say('PASS T01 the causal input scaling at bar t does not depend on scores at or after t')
# T02 fit rows
assert ROWS.bar.between(DEV_LO, DEV_HI - 1).all() and set(ROWS[ROWS.y == 1].lab.unique()) <= {1, 2, 3, 4, 5, 6}; say('PASS T02 fit rows lie inside VAL-3 (DEV bars); positives are old families only; no new-family row')
# T03 injected-series scaling uses the genuine series' statistics
assert (stats(MODELS[0], s0, 'GEN_L1')[0] is STAT[(MODELS[0], s0, 'GEN_L1')][0]); say('PASS T03 injected series are scaled with the statistics of the genuine reference series')
# T04 recall by a plain loop
rg = np.random.default_rng(0); sk, kk, ev0 = ESETS['CAL'][0]
for _ in range(5):
    al = rg.random(AOLEN) < 0.002; fast = event_hits(al, ev0); slow = np.array([al[r.t0:r.t0 + r.dur].any() for r in ev0.itertuples()]); assert (fast == slow).all()
say('PASS T04 event hit rule equals a plain loop')
# T05 stacker reproducibility
Xt, yt = XY(ROWS); f1 = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=1000, class_weight='balanced')).fit(Xt, yt); f2 = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=1000, class_weight='balanced')).fit(Xt, yt)
assert np.allclose(f1.decision_function(Xt[:500]), f2.decision_function(Xt[:500])); say('PASS T05 same data and settings give the same stacker')

PASS T01 the causal input scaling at bar t does not depend on scores at or after t
PASS T02 fit rows lie inside VAL-3 (DEV bars); positives are old families only; no new-family row
PASS T03 injected series are scaled with the statistics of the genuine reference series
PASS T04 event hit rule equals a plain loop
PASS T05 same data and settings give the same stacker


## 5 · Out-of-fold stackers on DEV

In [ ]:
NB = 4; EDGES = [DEV_LO + k * (DEV_HI - DEV_LO) // NB for k in range(NB + 1)]; PURGE = 240
def make(kind, Cc):
    if kind == 'LR': return make_pipeline(StandardScaler(), LogisticRegression(C=Cc, max_iter=2000, class_weight='balanced'))
    return xgb.XGBClassifier(max_depth=2, n_estimators=200, learning_rate=0.05, subsample=0.8, tree_method='hist', n_jobs=NTHREAD, random_state=11, eval_metric='logloss')
def margin(mdl, X):
    if hasattr(mdl, 'decision_function'): return mdl.decision_function(X)
    return mdl.predict(X, output_margin=True)
def fit_model(kind, Cc, rows):
    X, y = XY(rows); m = make(kind, Cc)
    if kind == 'GBM': m.set_params(scale_pos_weight=float((y == 0).sum() / max(1, (y == 1).sum())))
    return m.fit(X, y)
CANDS = [('LR', 0.01), ('LR', 0.1), ('LR', 1.0), ('GBM', 0)]
KEYS = {s: ['GEN_L1'] + sorted(k for k in [kk for (ss, kk) in FX if ss == s] if k.startswith('VAL')) for s in SYMBOLS}
OOF = {}
for cand in CANDS:
    OOF[cand] = {(s, k): np.full(129600, np.nan) for s in SYMBOLS for k in KEYS[s]}
    for b_ in range(NB):
        lo, hi = EDGES[b_], EDGES[b_ + 1]; tr = ROWS[(ROWS.bar < lo - PURGE) | (ROWS.bar >= hi + PURGE)]; assert ((tr.bar < lo - PURGE) | (tr.bar >= hi + PURGE)).all(); mdl = fit_model(*cand, tr)
        for s in SYMBOLS:
            for k in KEYS[s]:
                X = FX[(s, k)][lo:hi]; ok = ~np.isnan(X).any(axis=1); o = np.full(hi - lo, np.nan); o[ok] = margin(mdl, X[ok]); OOF[cand][(s, k)][lo:hi] = o
say('out-of-fold fusion scores built for ' + str(len(CANDS)) + ' stacker candidates (4 blocks, 240-bar purge)'); say('PASS T06 every fold was fitted without the held-out block and its 240-bar purge zone')

out-of-fold fusion scores built for 4 stacker candidates (4 blocks, 240-bar purge)
PASS T06 every fold was fitted without the held-out block and its 240-bar purge zone


## 6 · Evaluation helpers

In [ ]:
def thr_roll(g, q): t = B.rolling_thr(g, q); return np.where(np.isnan(t), np.inf, t)
RTC = {}
def thr_cached(key, g, q):
    k = (key, q)
    if k not in RTC: RTC[k] = thr_roll(g, q)
    return RTC[k]
def qsel(key, g, bud, valid):
    for q in B.QGRID:
        if B.eps_per_day(g > thr_cached(key, g, q), valid) <= bud: return q
    return B.QGRID[-1]
# ---- DEV evaluation of a set of series {(s, key): score array (NaN outside DEV)} : returns alarms (T1c) and q per symbol
def dev_alarms(series, bud, tag):
    out = {}; gen = {}
    for s in SYMBOLS:
        g = series[(s, 'GEN_L1')]; q = qsel((tag, s), g, bud, VD); thr = thr_cached((tag, s), g, q); gen[s] = (g > thr)
        for k in KEYS[s][1:]: out[(s, k)] = series[(s, k)] > thr
    return out, gen, {s: qsel((tag, s), series[(s, 'GEN_L1')], bud, VD) for s in SYMBOLS}
SEL_DEV = (META['VAL'].t0.to_numpy() >= DEV_LO + 1440)
def dev_hits(al):
    return np.concatenate([event_hits(al[(s, key)], ev) for s, key, ev in ESETS['VAL']])
def trim(a): o = np.full(129600, np.nan); o[DEV_LO:DEV_HI] = a[DEV_LO:DEV_HI]; return o
RAW = {m: {(s, k): trim(SCR[(m, s)][k]) for s in SYMBOLS for k in KEYS[s]} for m in MODELS}
def dev_eps_of(gen): return float(np.mean([B.eps_per_day(gen[s], VD) for s in SYMBOLS]))
def dev_eval_single(m, bud):
    al, gen, q = dev_alarms(RAW[m], bud, 'S|' + m); return al, gen
def dev_eval_union(bud):
    parts = [dev_eval_single(m, bud) for m in MODELS]; al = {k: np.any([p[0][k] for p in parts], axis=0) for k in parts[0][0]}; gen = {s: np.any([p[1][s] for p in parts], axis=0) for s in SYMBOLS}; return al, gen

## 7 · DEV comparison and the pre-registered selection

In [ ]:
rows = []; DEVH = {}
for cand in CANDS:
    for bud in BUD:
        al, gen, q = dev_alarms(OOF[cand], bud, 'F|' + str(cand)); h = dev_hits(al); DEVH[(cand, bud)] = (h, dev_eps_of(gen))
        rows.append({'stacker': f'{cand[0]} C={cand[1]}' if cand[0] == 'LR' else 'GBM depth 2', 'budget': bud, 'DEV eps/day': dev_eps_of(gen), 'OOF recall 6 old': macro_of(h, 'VAL', OLD, SEL_DEV), 'OOF recall 2 new': macro_of(h, 'VAL', NEW, SEL_DEV), 'cand': cand})
ST = pd.DataFrame(rows)
for m in MODELS:
    for bud in BUD:
        al, gen = dev_eval_single(m, bud); h = dev_hits(al); DEVH[(m, bud)] = (h, dev_eps_of(gen)); ST = pd.concat([ST, pd.DataFrame([{'stacker': m + ' (single)', 'budget': bud, 'DEV eps/day': dev_eps_of(gen), 'OOF recall 6 old': macro_of(h, 'VAL', OLD, SEL_DEV), 'OOF recall 2 new': macro_of(h, 'VAL', NEW, SEL_DEV), 'cand': None}])], ignore_index=True)
for bud in BUD:
    al, gen = dev_eval_union(bud); h = dev_hits(al); DEVH[('union', bud)] = (h, dev_eps_of(gen)); ST = pd.concat([ST, pd.DataFrame([{'stacker': 'union of three (each at the budget)', 'budget': bud, 'DEV eps/day': dev_eps_of(gen), 'OOF recall 6 old': macro_of(h, 'VAL', OLD, SEL_DEV), 'OOF recall 2 new': macro_of(h, 'VAL', NEW, SEL_DEV), 'cand': None}])], ignore_index=True)
say('--- DEV (VAL-3), T1c, out-of-fold fusion versus single models and the union (the same DEV bars for everybody) ---'); say(ST.drop(columns='cand').round(3).to_string(index=False))
sub = ST[(ST.budget == PRIM) & ST.cand.notna()].copy(); best = sub['OOF recall 6 old'].max(); tied = sub[sub['OOF recall 6 old'] >= best - 0.05].copy(); tied['k'] = [(0 if c[0] == 'LR' else 1, c[1]) for c in tied.cand]; CHOSEN = tied.sort_values('k').iloc[0].cand
say(f'selected stacker (tolerance 0.05, simplest first): {CHOSEN}  (OOF recall {sub[sub.cand == CHOSEN]["OOF recall 6 old"].iloc[0]:.3f}; best {best:.3f})')

--- DEV (VAL-3), T1c, out-of-fold fusion versus single models and the union (the same DEV bars for everybody) ---
                            stacker  budget  DEV eps/day  OOF recall 6 old  OOF recall 2 new
                          LR C=0.01       1        0.731             0.674             0.647
                          LR C=0.01       2        1.558             0.732             0.784
                          LR C=0.01       4        3.423             0.801             0.834
                           LR C=0.1       1        0.712             0.674             0.640
                           LR C=0.1       2        1.558             0.732             0.784
                           LR C=0.1       4        3.462             0.801             0.834
                           LR C=1.0       1        0.712             0.674             0.633
                           LR C=1.0       2        1.538             0.732             0.784
                           LR C=1.0       4      

## 8 · Final stacker, CAL-3 evaluation, hypotheses

In [ ]:
FM = fit_model(*CHOSEN, ROWS)
if CHOSEN[0] == 'LR':
    sc_, lr_ = FM.steps[0][1], FM.steps[1][1]; COEF = pd.DataFrame({'feature': FNAMES, 'coef (per SD)': lr_.coef_[0]}); say('logistic stacker (standardised inputs): intercept ' + f'{lr_.intercept_[0]:.3f}'); say(COEF.round(3).to_string(index=False))
FS = {}
for s in SYMBOLS:
    for k in ['GEN_AO', 'CAL_r0']:
        X = FX[(s, k)]; ok = ~np.isnan(X).any(axis=1); o = np.full(AOLEN, np.nan); o[ok] = margin(FM, X[ok]); FS[(s, k)] = o
# thresholds: q per budget from the OUT-OF-FOLD genuine DEV fused score; rolling threshold from the genuine CAL fused series (T1c)
QF = {bud: dev_alarms(OOF[CHOSEN], bud, 'F|' + str(CHOSEN))[2] for bud in BUD}
def cal_alarm_fusion(bud):
    out = {}; gen = {}
    for s in SYMBOLS:
        thr = thr_roll(FS[(s, 'GEN_AO')], QF[bud][s]); gen[s] = FS[(s, 'GEN_AO')] > thr; out[s] = FS[(s, 'CAL_r0')] > thr
    return out, gen
def cal_alarm_single(m, bud):
    out = {}; gen = {}
    for s in SYMBOLS:
        g = SCR[(m, s)]['GEN_L1']; q = B.t1_q(g, L1V, bud); thr = thr_roll(SCR[(m, s)]['GEN_AO'], q); gen[s] = SCR[(m, s)]['GEN_AO'] > thr; out[s] = SCR[(m, s)]['CAL_r0'] > thr
    return out, gen
def cal_hits(al): return np.concatenate([event_hits(al[s], ev) for s, key, ev in ESETS['CAL']])
def cal_eps(gen): return float(np.mean([B.eps_per_day(gen[s], CALV) for s in SYMBOLS]))
CH = {}; CE = {}
for bud in BUD:
    a, g = cal_alarm_fusion(bud); CH[('fusion', bud)] = cal_hits(a); CE[('fusion', bud)] = cal_eps(g)
    parts = []
    for m in MODELS: a, g = cal_alarm_single(m, bud); CH[(m, bud)] = cal_hits(a); CE[(m, bud)] = cal_eps(g); parts.append((a, g))
    CH[('union', bud)] = np.any([CH[(m, bud)] for m in MODELS], axis=0); CE[('union', bud)] = cal_eps({s: np.any([p[1][s] for p in parts], axis=0) for s in SYMBOLS})
# T07 agreement with notebook 31 (single-model T1c recall at the primary budget)
n31 = pd.read_csv(OUT/'cal3_recall.csv')
for m in MODELS:
    r = n31[(n31.model == m) & (n31.norm == FNORM[m]) & (n31.rule == 'T1c') & (n31.budget == PRIM)]['recall 6 old'].iloc[0]; assert abs(r - macro_of(CH[(m, PRIM)], 'CAL', OLD)) < 1e-9, (m, r)
say('PASS T07 the single-model CAL-3 recall recomputed here equals notebook 31 (T1c, budget 2)')
crow = []
for name in ['fusion'] + MODELS + ['union']:
    for bud in BUD:
        h = CH[(name, bud)]; b_o = boot_paired([h], 'CAL', OLD)[:, 0]; b_n = boot_paired([h], 'CAL', NEW)[:, 0]
        crow.append({'method': name, 'budget': bud, 'CAL genuine eps/day': CE[(name, bud)], 'recall 6 old': macro_of(h, 'CAL', OLD), 'old CI': ci(b_o), 'recall 2 new': macro_of(h, 'CAL', NEW), 'new CI': ci(b_n), 'macro 8': macro_of(h, 'CAL', ALLF)})
CALT = pd.DataFrame(crow); say(''); say('--- CAL-3 (200 events per symbol): fusion v0 versus the final single models and their union (T1c) ---'); say(CALT.round(3).to_string(index=False))
FAMTAB = pd.DataFrame({name: fam_rates(CH[(name, PRIM)], 'CAL') for name in ['fusion'] + MODELS + ['union']}); FAMTAB.index = [B.FAMS[i] for i in FAMTAB.index]; say(''); say('--- CAL-3 per family, budget 2/day ---'); say(FAMTAB.round(2).to_string())
# hypotheses
single = {m: macro_of(CH[(m, PRIM)], 'CAL', OLD) for m in MODELS}; bm = max(single, key=single.get)
fu, un = macro_of(CH[('fusion', PRIM)], 'CAL', OLD), macro_of(CH[('union', PRIM)], 'CAL', OLD)
HF1 = bool(fu >= un - 0.05 and CE[('fusion', PRIM)] <= 0.6 * CE[('union', PRIM)])
d = boot_paired([CH[('fusion', PRIM)], CH[(bm, PRIM)]], 'CAL', OLD); diff = d[:, 0] - d[:, 1]; HF2 = bool(fu - single[bm] >= 0.03 and np.percentile(diff, 2.5) > 0)
fn_, bn_ = macro_of(CH[('fusion', PRIM)], 'CAL', NEW), max(macro_of(CH[(m, PRIM)], 'CAL', NEW) for m in MODELS); HF3 = bool(fn_ >= bn_ - 0.05)
HYP = {'H-F1 burden (fusion within 0.05 of the union recall, genuine CAL episodes/day <= 60% of the union)': {'fusion recall': fu, 'union recall': un, 'fusion eps/day': CE[('fusion', PRIM)], 'union eps/day': CE[('union', PRIM)], 'holds': HF1},
       'H-F2 beats best single by >= 0.03 with paired 95% CI > 0': {'best single': bm, 'best single recall': single[bm], 'fusion recall': fu, 'difference 95% CI': ci(diff), 'holds': HF2},
       'H-F3 unseen families within 0.05 of the best single': {'fusion new2': fn_, 'best single new2': bn_, 'holds': HF3}}
say(''); say('--- hypotheses (pre-registered, CAL-3, budget 2/day, T1c) ---'); [say(f'{k}: {v}') for k, v in HYP.items()]

logistic stacker (standardised inputs): intercept -1.007
            feature  coef (per SD)
      z AE-ctx fmax         -0.072
       z XGBoost v2          0.380
   z Transformer v2          0.531
              z max          0.409
             z mean          0.388
   max5 AE-ctx fmax         -0.023
    max5 XGBoost v2          0.382
max5 Transformer v2          0.468
PASS T07 the single-model CAL-3 recall recomputed here equals notebook 31 (T1c, budget 2)

--- CAL-3 (200 events per symbol): fusion v0 versus the final single models and their union (T1c) ---
        method  budget  CAL genuine eps/day  recall 6 old    old CI  recall 2 new    new CI  macro 8
        fusion       1                0.783         0.737 0.70-0.77          0.68 0.58-0.77    0.722
        fusion       2                1.567         0.780 0.74-0.81          0.74 0.65-0.83    0.770
        fusion       4                3.317         0.807 0.77-0.84          0.80 0.72-0.87    0.805
   AE-ctx fmax       1         

[None, None, None]

## 9 · Per-model contribution

In [ ]:
CONTRIB = None
if CHOSEN[0] == 'LR':
    sc_, lr_ = FM.steps[0][1], FM.steps[1][1]; w = lr_.coef_[0] / sc_.scale_; crows = []
    for s, key, ev in ESETS['CAL']:
        X = FX[(s, key)]; con = (X - sc_.mean_[None, :]) * w[None, :]            # additive contribution of every input to the logit
        grp = {m: [i for i, n_ in enumerate(FNAMES) if n_ in ('z ' + m, 'max5 ' + m)] for m in MODELS}; grp['shared (max, mean)'] = [FNAMES.index('z max'), FNAMES.index('z mean')]
        for r_ in ev.itertuples():
            seg = con[r_.t0:r_.t0 + r_.dur]; crows.append({'family': B.FAMS[int(r_.family)], **{g: float(np.nanmean(seg[:, ix].sum(axis=1))) for g, ix in grp.items()}})
    CONTRIB = pd.DataFrame(crows).groupby('family').mean(); say(''); say('--- mean contribution to the fused logit during CAL-3 events (additive, exact for the logistic stacker; larger = pushes the score up) ---'); say(CONTRIB.round(2).to_string())
else: say('(the selected stacker is a boosted-tree model: additive contributions are not exact; SHAP is added in the next protocol)')


--- mean contribution to the fused logit during CAL-3 events (additive, exact for the logistic stacker; larger = pushes the score up) ---
                           AE-ctx fmax  XGBoost v2  Transformer v2  shared (max, mean)
family                                                                                
abnormal_volume_burst            -0.09        0.20            0.36                0.49
fake_breakout_like               -0.06        1.24            1.46                1.25
liquidity_stress_proxy           -0.18        1.22            1.76                1.56
momentum_ignition_like           -0.03        0.39            0.36                0.38
pump_and_dump_like               -0.06        0.82            1.32                0.99
slow_accumulation_like            0.01        0.59            0.49                0.46
trade_flow_imbalance             -0.01        1.10            1.14                0.99
wash_like_aggregate_proxy        -0.07        1.56            2.17            

## 10 · Save and declare

In [ ]:
ST.drop(columns='cand').to_csv(OUT/'fusion_v0_dev_oof_table.csv', index=False); CALT.to_csv(OUT/'fusion_v0_cal3_table.csv', index=False); FAMTAB.to_csv(OUT/'fusion_v0_cal3_by_family.csv')
if CONTRIB is not None: CONTRIB.to_csv(OUT/'fusion_v0_contributions.csv')
joblib.dump(FM, FUS/'fusion_v0_stacker.joblib')
for s in SYMBOLS: np.savez_compressed(OUT/'scores'/f'fusion_v0_{s}.npz', GEN_AO=FS[(s, 'GEN_AO')], CAL_r0=FS[(s, 'CAL_r0')], **{'OOF_' + k: OOF[CHOSEN][(s, k)] for k in KEYS[s]})
DECLF = {'status': 'DECLARED after the DEV out-of-fold selection and the CAL-3 evaluation; TEST-3 is evaluated in notebook 32 with this frozen stacker; no change before then.', 'declared_utc': datetime.datetime.utcnow().isoformat(timespec='seconds') + 'Z',
         'protocol_F0_sha256': PH, 'declared_final_set_sha16': sha_file(OUT/'declared_final_set_v1.json'), 'stacker': {'kind': CHOSEN[0], 'C': CHOSEN[1]}, 'inputs': FNAMES, 'input_models': {m: FNORM[m] for m in MODELS},
         'input_transform': {'AE-ctx fmax': 'log1p(score)', 'XGBoost v2': 'margin', 'Transformer v2': 'margin'}, 'scaling': 'causal rolling 7-day median and IQR/1.349 of the model own transformed score (min 1,440 bars; DEV genuine median/scale fallback; scale floor 0.1 x DEV scale), clip +-15',
         'q_T1_by_budget': {str(b): QF[b] for b in BUD}, 'primary_budget': PRIM, 'hypotheses': HYP, 'stacker_file': 'fusion/fusion_v0_stacker.joblib'}
if CHOSEN[0] == 'LR': DECLF['logistic'] = {'scaler_mean': sc_.mean_.tolist(), 'scaler_scale': sc_.scale_.tolist(), 'coef': lr_.coef_[0].tolist(), 'intercept': float(lr_.intercept_[0])}
(OUT/'declared_fusion_v0.json').write_text(json.dumps(DECLF, indent=1, default=str)); say('saved tables, stacker, fused scores and declared_fusion_v0.json')
block = '\n'.join(LINES); (OUT/'b2_step6_share_block.txt').write_text(block)
print('\n===== SHARE BLOCK (attach models_b2_v1/b2_step6_share_block.txt, or copy everything below) =====\n' + block[-14000:])

saved tables, stacker, fused scores and declared_fusion_v0.json

===== SHARE BLOCK (attach models_b2_v1/b2_step6_share_block.txt, or copy everything below) =====
protocol B2 intact, sha256 443f99d5faf5929f; fusion protocol F0 sha256 b1d84483c102bf65
library sha256 1b0b46630763c944
data ready: models {'AE-ctx fmax': 'N1', 'XGBoost v2': 'N0', 'Transformer v2': 'N1'}; DEV evaluation bars 37440; CAL events per symbol {'BTCUSDT': 200, 'ETHUSDT': 200}
fusion features ['z AE-ctx fmax', 'z XGBoost v2', 'z Transformer v2', 'z max', 'z mean', 'max5 AE-ctx fmax', 'max5 XGBoost v2', 'max5 Transformer v2']; example (BTC, genuine DEV) median [-0.    0.04  0.01  0.74  0.09  1.53  0.79  0.79]
fit rows: 9286 positive bars (old families), 60015 clean negatives; new families excluded
PASS T01 the causal input scaling at bar t does not depend on scores at or after t
PASS T02 fit rows lie inside VAL-3 (DEV bars); positives are old families only; no new-family row
PASS T03 injected series are scaled with th